# Data Understanding — Brazilian E-commerce Dataset

## 1. Project Overview

В данном проекте проводится анализ данных бразильского маркетплейса Olist.

Цель этапа Data Understanding:

- изучить структуру исходного датасета;
- определить бизнес-сущности и назначение таблиц;
- проверить качество данных;
- определить ключи для объединения таблиц;
- подготовить данные для дальнейшего анализа.

## 2. Dataset Loading

Датасет состоит из 9 связанных таблиц, каждая из которых описывает отдельную часть бизнес-процесса:

- клиенты;
- заказы;
- товары;
- платежи;
- отзывы;
- продавцы;
- географические данные;
- категории товаров.

Все таблицы загружаются из локальной директории `data/`.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [ ]:
DATA_PATH = "../data/"

customers = pd.read_csv(
    DATA_PATH + "olist_customers_dataset.csv"
)

orders = pd.read_csv(
    DATA_PATH + "olist_orders_dataset.csv"
)

items = pd.read_csv(
    DATA_PATH + "olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    DATA_PATH + "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    DATA_PATH + "olist_order_reviews_dataset.csv"
)

products = pd.read_csv(
    DATA_PATH + "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    DATA_PATH + "olist_sellers_dataset.csv"
)

geo = pd.read_csv(
    DATA_PATH + "olist_geolocation_dataset.csv"
)

categories = pd.read_csv(
    DATA_PATH + "product_category_name_translation.csv"
)

datasets = {
    "customers": customers,
    "orders": orders,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geo": geo,
    "categories": categories
}

FileNotFoundError: [Errno 2] No such file or directory: '../data/olist_customers_dataset.csv'

## 3. Dataset Structure

На данном этапе оценивается размер каждой таблицы:

- количество строк;
- количество признаков;
- первые записи для понимания структуры данных.

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.shape)
    print(df.head())


customers
(99441, 5)
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  

orders
(99441, 8)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9e

### Dataset Summary

Всего было загружено:

- 9 таблиц;
- 99441 заказ;
- 96096 уникальных клиентов;
- 32951 товар;
- 3095 продавцов.

In [ ]:
for name, df in datasets.items():
    df.info()
    print('')
    

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 

## 4. Data Quality Assessment

Проверяются:

- типы данных;
- количество пропусков;
- количество уникальных значений.

In [ ]:
for name, df in datasets.items():
    print(name)
    print(df.isnull().sum())
    print('')

customers
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

orders
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

items
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

payments
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

reviews
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_commen

### Data Quality Notes

Обнаружены следующие особенности данных:

- `orders`:
  - часть заказов не имеет даты доставки;
  - вероятная причина — отменённые или незавершённые заказы.

- `reviews`:
  - текстовые комментарии отсутствуют у части клиентов;
  - для количественного анализа используются оценки `review_score`;
  - текстовые отзывы не рассматриваются в текущем проекте.

- При расчёте метрик доставки необходимо учитывать только заказы с заполненными датами доставки.

In [ ]:
for name, df in datasets.items():
    print(name)
    print(df.nunique())
    print('')

customers
customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64

orders
order_id                         99441
customer_id                      99441
order_status                         8
order_purchase_timestamp         98875
order_approved_at                90733
order_delivered_carrier_date     81018
order_delivered_customer_date    95664
order_estimated_delivery_date      459
dtype: int64

items
order_id               98666
order_item_id             21
product_id             32951
seller_id               3095
shipping_limit_date    93318
price                   5968
freight_value           6999
dtype: int64

payments
order_id                99440
payment_sequential         29
payment_type                5
payment_installments       24
payment_value           29077
dtype: int64

reviews
review_id                  98410
order_id                   98673
rev

## 5. Tables Description

Описание бизнес-роли каждой таблицы:

| Table | Description |
|---|---|
| orders | Информация о заказах |
| customers | Клиенты |
| items | Товары внутри заказа |
| payments | Платежи |
| reviews | Отзывы клиентов |
| sellers | Продавцы |
| products | Каталог товаров |
| geo | География клиентов и продавцов |
| categories | Категории товаров |

In [ ]:
for name, df in datasets.items():
    print(name)
    print(df.columns)
    print('')

customers
Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')

orders
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

items
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')

payments
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')

reviews
Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')

products
Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_phot

## 6. Data Relationships

Основной объект анализа — заказ (`orders`).

Большинство таблиц связываются через идентификатор заказа:

- товары заказа;
- платежи;
- отзывы.

Дополнительные связи используются для анализа клиентов, продавцов и категорий товаров.

| Таблица | Связь |
|---|---|
| customers → orders | customer_id |
| customers → geo | customer_zip_code_prefix |
| sellers → geo | seller_zip_code_prefix |
| orders → items | order_id |
| orders → payments | order_id |
| orders → reviews | order_id |
| items → products | product_id |
| items → sellers | seller_id |
| products → categories | product_category_name |

### Main Analytical Flow

Основная цепочка объединения данных:

customers → orders → items → products → categories

Дополнительные таблицы:

orders → payments  
orders → reviews  
items → sellers  
geo → customers/sellers

In [ ]:
dfs = []
for name, df in datasets.items():
    dfs.append([name, df.shape[0], df.shape[1]])
dfs = pd.DataFrame(dfs, columns=["table", "rows", "columns"])
print(dfs)

        table     rows  columns
0   customers    99441        5
1      orders    99441        8
2       items   112650        7
3    payments   103886        5
4     reviews    99224        7
5    products    32951        9
6     sellers     3095        4
7         geo  1000163        5
8  categories       71        2


## Business Entities Summary

Всего получено 9 таблиц.

Основные талблицы:

- orders — центральная таблица заказов;
- customers — информация о клиентах;
- items — товары внутри заказа;
- payments — информация об оплатах;
- reviews — оценки клиентов;
- sellers — продавцы;
- products — характеристики товаров.

## 7. Key Columns

Основные идентификаторы:

| Column | Description |
|-|-|
| order_id | идентификатор заказа |
| customer_id | клиент внутри заказа |
| customer_unique_id | уникальный пользователь |
| product_id | товар |
| seller_id | продавец |
|product_category_name |	категория товара |
|customer_zip_code_prefix |	регион клиента |
|seller_zip_code_prefix |	регион продавца |

## 8. Limitations

Ограничения данных:

- датасет содержит исторические данные за ограниченный период;
- отсутствует информация о маркетинговых каналах привлечения клиентов;
- часть бизнес-процессов представлена неполностью из-за пропусков;
- данные не позволяют напрямую оценить прибыльность бизнеса без информации о затратах.

# 9. Conclusions

На этапе Data Understanding:

- загружено 9 связанных таблиц;
- определены основные бизнес-сущности;
- таблица `orders` выбрана как центральная таблица анализа;
- определены ключи для объединения данных;
- выявлены особенности качества данных;
- сформирована схема взаимосвязей таблиц;
- определены основные сущности и ключевые поля;
- подготовлена структура для построения аналитических витрин.

Следующий этап:

Data Cleaning & Preparation — обработка пропусков, типов данных и подготовка аналитических таблиц.